# Notebook 01 — Data Collection

## Objectives
- Fetch the used-car dataset from Kaggle (our data **endpoint**) using the Kaggle API.
- Combine the per-manufacturer files into a single dataset, handling known data quirks.
- Save the combined dataset and take a first look (CRISP-DM: Data Understanding).

## Inputs
- Kaggle authentication token, read automatically from `~/.kaggle/access_token`
  (kept in the home directory, **never committed** to the repo).
- Kaggle dataset: `adityadesai13/used-car-dataset-ford-and-mercedes`
  (100,000 UK Used Car Data set, licence **CC0: Public Domain**).

## Outputs
- Raw CSVs saved to `inputs/datasets/raw/`.
- A single combined dataset saved to `outputs/datasets/collection/used_cars.csv`.

> Satisfies Pass **7.1** (data collection from an endpoint via Jupyter Notebook)
> and contributes to Pass **1.1** (Business/Data Understanding under CRISP-DM).

---
## 1. Imports and working directory

In [11]:
import os
import pandas as pd

# Make sure we run from the project root, whether the notebook is opened
# from the root or from inside the jupyter_notebooks/ folder.
if os.path.basename(os.getcwd()) == "jupyter_notebooks":
    os.chdir(os.path.dirname(os.getcwd()))
print("Working directory:", os.getcwd())

Working directory: /workspaces/used-car-price-predictor


---
## 2. Fetch the dataset from Kaggle (the endpoint)

The Kaggle client reads the API token automatically from `~/.kaggle/access_token`.
We download the dataset and unzip it straight into `inputs/datasets/raw/`.

In [12]:
DATASET = "adityadesai13/used-car-dataset-ford-and-mercedes"
RAW_DIR = "inputs/datasets/raw"
os.makedirs(RAW_DIR, exist_ok=True)

!kaggle datasets download -d {DATASET} -p {RAW_DIR} --unzip

Dataset URL: https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes
License(s): CC0-1.0
100%|█████████████████████████████████████| 1.10M/1.10M [00:00<00:00, 2.47MB/s]



In [13]:
# Confirm what was downloaded
sorted(f for f in os.listdir(RAW_DIR) if f.endswith(".csv"))

['audi.csv',
 'bmw.csv',
 'cclass.csv',
 'focus.csv',
 'ford.csv',
 'hyundi.csv',
 'merc.csv',
 'skoda.csv',
 'toyota.csv',
 'unclean cclass.csv',
 'unclean focus.csv',
 'vauxhall.csv',
 'vw.csv']

---
## 3. Combine the manufacturer files

The dataset ships one CSV per manufacturer. To build a multi-brand pricing tool we
combine them into one dataframe with a `manufacturer` column. Before combining we:

- **Skip** the model-subset files (`cclass.csv`, `focus.csv`) — their cars already
  appear inside `merc.csv` and `ford.csv`, so including them would duplicate rows.
- **Skip** the `unclean ...` files — they are the raw, pre-cleaned versions.
- **Standardise columns** — the Hyundai file uses `tax(£)`; we rename it to `tax`
  so every file lines up.

In [14]:
# Files that must NOT be combined (duplicates / pre-cleaned versions)
SKIP = {"cclass.csv", "focus.csv", "unclean cclass.csv", "unclean focus.csv"}

# Tidy display names for each manufacturer file
MANUFACTURER_NAMES = {
    "audi": "Audi", "bmw": "BMW", "ford": "Ford", "hyundi": "Hyundai",
    "merc": "Mercedes", "skoda": "Skoda", "toyota": "Toyota",
    "vauxhall": "Vauxhall", "vw": "Volkswagen",
}

frames = []
for f in sorted(os.listdir(RAW_DIR)):
    if f.endswith(".csv") and f not in SKIP:
        tmp = pd.read_csv(os.path.join(RAW_DIR, f))
        tmp.columns = [col.strip() for col in tmp.columns]   # drop stray whitespace
        tmp = tmp.rename(columns={"tax(£)": "tax"})          # align the Hyundai quirk
        key = f.replace(".csv", "")
        tmp["manufacturer"] = MANUFACTURER_NAMES.get(key, key.title())
        frames.append(tmp)

df = pd.concat(frames, ignore_index=True)
print(f"Combined {len(frames)} manufacturer files into one dataset.")
df.shape

Combined 9 manufacturer files into one dataset.


(99187, 10)

---
## 4. First look (Data Understanding)

A quick inspection: column types, a sample of rows, summary statistics, and a
missing-value check. We will write our conclusions from these outputs into the
README and the Project Summary dashboard page.

In [15]:
df.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,manufacturer
0,A1,2017,12500,Manual,15735,Petrol,150,55.4,1.4,Audi
1,A6,2016,16500,Automatic,36203,Diesel,20,64.2,2.0,Audi
2,A1,2016,11000,Manual,29946,Petrol,30,55.4,1.4,Audi
3,A4,2017,16800,Automatic,25952,Diesel,145,67.3,2.0,Audi
4,A3,2019,17300,Manual,1998,Petrol,145,49.6,1.0,Audi


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99187 entries, 0 to 99186
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   model         99187 non-null  str    
 1   year          99187 non-null  int64  
 2   price         99187 non-null  int64  
 3   transmission  99187 non-null  str    
 4   mileage       99187 non-null  int64  
 5   fuelType      99187 non-null  str    
 6   tax           99187 non-null  int64  
 7   mpg           99187 non-null  float64
 8   engineSize    99187 non-null  float64
 9   manufacturer  99187 non-null  str    
dtypes: float64(2), int64(4), str(4)
memory usage: 10.0 MB


In [17]:
df.describe()

,year,price,mileage,tax,mpg,engineSize
count,99187.000000,99187.000000,99187.000000,99187.000000,99187.000000,99187.000000
mean,2017.087723,16805.347656,23058.914213,120.299838,55.166825,1.663280
std,2.123934,9866.773417,21148.523721,63.150926,16.138522,0.557646
min,1970.000000,450.000000,1.000000,0.000000,0.300000,0.000000
25%,2016.000000,9999.000000,7425.000000,125.000000,47.100000,1.200000
50%,2017.000000,14495.000000,17460.000000,145.000000,54.300000,1.600000
75%,2019.000000,20870.000000,32339.000000,145.000000,62.800000,2.000000
max,2060.000000,159999.000000,323000.000000,580.000000,470.800000,6.600000


In [18]:
# Missing values per column
df.isna().sum()

model           0
year            0
price           0
transmission    0
mileage         0
fuelType        0
tax             0
mpg             0
engineSize      0
manufacturer    0
dtype: int64

In [19]:
# How many rows per manufacturer?
df["manufacturer"].value_counts()

manufacturer
Ford          17965
Volkswagen    15157
Vauxhall      13632
Mercedes      13119
BMW           10781
Audi          10668
Toyota         6738
Skoda          6267
Hyundai        4860
Name: count, dtype: int64

---
## 5. Save the combined dataset

In [20]:
OUT_DIR = "outputs/datasets/collection"
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "used_cars.csv")
df.to_csv(out_path, index=False)
print("Saved:", out_path, "| rows:", len(df), "| columns:", df.shape[1])

Saved: outputs/datasets/collection/used_cars.csv | rows: 99187 | columns: 10


## 6. Conclusions and next steps
The combined dataset has **99,187 rows and 10 columns** across **nine manufacturers**
(Ford, Volkswagen, Vauxhall, Mercedes, BMW, Audi, Toyota, Skoda, Hyundai). Each row is
one used-car listing.

- **Target:** `price` (£) — the value the model will predict.
- **Features:** `manufacturer`, `model`, `year`, `transmission`, `mileage`,
  `fuelType`, `tax`, `mpg`, `engineSize`.
- **No missing values** — every column is fully populated.
- **Data-quality issues to handle in cleaning** (from `describe()`):
  - `year` reaches **2060** — impossible future date(s).
  - `mpg` spans **0.3 to 470.8** — both physically implausible.
  - `engineSize` has a minimum of **0.0** — likely electric cars or missing-as-zero.
  - `price` minimum of **£450** is suspiciously low — possible data-entry errors.

Next: **Notebook 02 — Data Cleaning** (CRISP-DM: Data Preparation).